# DeepSeek-OCR-2: Visual Causal Flow

This notebook runs DeepSeek-OCR-2 on Google Colab with GPU.

**Paper:** [arXiv:2601.20552](https://arxiv.org/abs/2601.20552)

**Requirements:**
- GPU runtime (T4 or better)
- ~15GB GPU memory for inference

## 1. Check GPU and Setup Environment

In [ ]:
# Check GPU
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Install dependencies
!pip install -q transformers>=4.45.0 accelerate>=0.34.0 pillow

# For vLLM (optional, for faster inference)
# !pip install vllm==0.8.5

## 2. Load DeepSeek-OCR-2 Model

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch

MODEL_NAME = "deepseek-ai/DeepSeek-OCR-2"

print(f"Loading model: {MODEL_NAME}")
print("This may take a few minutes...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
model.eval()

print("\n✓ Model loaded successfully!")

## 3. Upload Test Images

In [ ]:
from google.colab import files
import os

# Create directory for images
os.makedirs("test-images", exist_ok=True)

print("Upload your test images:")
uploaded = files.upload()

# Move uploaded files to test-images directory
for filename in uploaded.keys():
    os.rename(filename, f"test-images/{filename}")
    print(f"  ✓ Saved: test-images/{filename}")

print(f"\nTotal images: {len(os.listdir('test-images'))}")

## 4. OCR Functions

In [ ]:
import time
import json
from datetime import datetime
from pathlib import Path
from PIL import Image

# OCR Prompts
PROMPTS = {
    "standard": "<image>\nConvert this image to text.",
    "markdown": "<image>\n<|grounding|>Convert this document to markdown format.",
    "detailed": "<image>\nExtract all text from this image, including handwritten content. Preserve the layout.",
    "vietnamese": "<image>\nTrích xuất tất cả văn bản từ hình ảnh này, bao gồm cả chữ viết tay."
}

def perform_ocr(image_path: str, prompt_type: str = "detailed") -> dict:
    """Perform OCR using DeepSeek-OCR-2"""
    prompt = PROMPTS.get(prompt_type, PROMPTS["detailed"])

    start_time = time.time()
    try:
        image = Image.open(image_path).convert("RGB")

        result = model.infer(
            tokenizer,
            prompt=prompt,
            image_file=image_path
        )

        elapsed_time = time.time() - start_time

        return {
            "success": True,
            "text": result,
            "elapsed_time": elapsed_time,
            "model": MODEL_NAME,
            "prompt_type": prompt_type,
            "image_size": image.size
        }
    except Exception as e:
        elapsed_time = time.time() - start_time
        return {
            "success": False,
            "error": str(e),
            "elapsed_time": elapsed_time,
            "model": MODEL_NAME,
            "prompt_type": prompt_type
        }

print("✓ OCR functions defined")

## 5. Run OCR on All Images

In [ ]:
IMAGE_DIR = Path("test-images")
image_extensions = {".png", ".jpg", ".jpeg", ".webp"}

image_files = [f for f in IMAGE_DIR.iterdir()
               if f.suffix.lower() in image_extensions]

print(f"Found {len(image_files)} images")
print("=" * 60)

all_results = {
    "detailed": {},
    "vietnamese": {},
    "markdown": {}
}

for image_path in sorted(image_files):
    print(f"\n📷 Processing: {image_path.name}")

    for prompt_type in ["detailed", "vietnamese", "markdown"]:
        result = perform_ocr(str(image_path), prompt_type)
        all_results[prompt_type][image_path.name] = {
            **result,
            "timestamp": datetime.now().isoformat()
        }

        if result["success"]:
            print(f"  ✓ {prompt_type}: {result['elapsed_time']:.2f}s, {len(result['text'])} chars")
        else:
            print(f"  ✗ {prompt_type}: {result['error']}")

print("\n" + "=" * 60)
print("✓ All images processed!")

## 6. View Results

In [ ]:
# Display results for each image
for image_name in all_results["detailed"].keys():
    print("=" * 80)
    print(f"IMAGE: {image_name}")
    print("=" * 80)

    result = all_results["detailed"][image_name]
    if result["success"]:
        print(f"\nTime: {result['elapsed_time']:.2f}s")
        print(f"Size: {result.get('image_size', 'N/A')}")
        print(f"\nExtracted text:\n{'-'*40}")
        print(result["text"])
    else:
        print(f"Error: {result['error']}")

    print()

## 7. Save Results

In [ ]:
os.makedirs("results", exist_ok=True)

# Save each prompt type results
for prompt_type, results in all_results.items():
    filename = f"results/deepseek_{prompt_type}.json"
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"✓ Saved: {filename}")

print("\nDownloading results...")

# Zip results for download
!zip -r deepseek_ocr_results.zip results/

files.download("deepseek_ocr_results.zip")

## 8. Quick Test with Single Image

In [ ]:
# Quick test with a single image
# You can modify this cell to test specific images

from IPython.display import display

# Upload single image
print("Upload a single image for quick test:")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\nProcessing: {filename}")

    # Display image
    img = Image.open(filename)
    display(img.resize((400, int(400 * img.height / img.width))))

    # Run OCR
    result = perform_ocr(filename, "detailed")

    print(f"\nTime: {result['elapsed_time']:.2f}s")
    print(f"\nExtracted text:\n{'='*40}")
    print(result["text"] if result["success"] else f"Error: {result['error']}")

## 9. Compare with Gemini (Optional)

If you have Gemini API key, you can compare results here.

In [ ]:
# Install Gemini SDK
!pip install -q google-generativeai

import google.generativeai as genai

# Set your API key
GOOGLE_API_KEY = ""  # Add your key here

if GOOGLE_API_KEY:
    genai.configure(api_key=GOOGLE_API_KEY)
    gemini_model = genai.GenerativeModel("gemini-2.0-flash")

    def gemini_ocr(image_path: str) -> dict:
        start_time = time.time()
        try:
            image = Image.open(image_path)
            prompt = "Extract all text from this image, including handwritten content. Preserve the layout."
            response = gemini_model.generate_content([prompt, image])
            elapsed_time = time.time() - start_time
            return {
                "success": True,
                "text": response.text,
                "elapsed_time": elapsed_time
            }
        except Exception as e:
            return {
                "success": False,
                "error": str(e),
                "elapsed_time": time.time() - start_time
            }

    print("✓ Gemini configured. Use gemini_ocr(image_path) to compare.")
else:
    print("Add your GOOGLE_API_KEY to enable Gemini comparison")